# 03 — Chart Anatomy & Styling
**Goal:** Master the parts of a chart (spines, ticks, gridlines, labels,
legends) and the styling system (`rcParams` and style sheets) so every chart
you ship looks intentional.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print('matplotlib', plt.matplotlib.__version__)

## 1. The anatomy of a polished chart

Every element you see has a name you can target. Here is the vocabulary you'll
use for the rest of the course.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = pd.date_range('2024-01-01', periods=12, freq='MS')
y = (np.random.rand(12).cumsum() + 5)
ax.plot(x, y, marker='o', color='steelblue', label='MRR ($M)')
ax.fill_between(x, y*0.9, y*1.1, color='steelblue', alpha=0.15)

ax.set_title('Monthly Recurring Revenue', fontsize=14, weight='bold', loc='left', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('MRR (USD millions)', fontsize=11)
ax.legend(loc='upper left', frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='gray', alpha=0.3)
ax.tick_params(axis='both', labelsize=10)

ax.annotate('Product launch',
            xy=(x[5], y[5]), xytext=(x[5], y[5]+1.2),
            ha='center', fontsize=9, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson', lw=1))
plt.tight_layout()
plt.show()

## 2. Spines — the frame around the plot

The four spines (`top`, `right`, `bottom`, `left`) are the most obvious
stylistic handle. Removing the top and right is the single biggest change
between "matplotlib default" and "looks like a real report".

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
x = np.linspace(0, 10, 100)

axes[0].plot(x, np.sin(x), color='steelblue')
axes[0].set_title('Default — all four spines')

axes[1].plot(x, np.sin(x), color='steelblue')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].set_title('No top/right (Tufte-style)')

axes[2].plot(x, np.sin(x), color='steelblue')
for s in axes[2].spines.values():
    s.set_color('gray'); s.set_linewidth(0.8)
axes[2].set_title('Lighter frame')
plt.tight_layout()
plt.show()

## 3. Ticks and tick labels

You can change tick position, format, density, and label content. Useful
methods:

- `ax.set_xticks([...])` / `ax.set_yticks([...])` — explicit positions
- `ax.set_xticklabels([...])` / `ax.set_yticklabels([...])` — labels
- `ax.tick_params(...)` — color, size, direction, label rotation
- `ax.xaxis.set_major_formatter(...)` — number / date / percent format
- `ax.xaxis.set_major_locator(...)` — placement (e.g. every 3 months)

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(9, 3.5))
x = pd.date_range('2023-01-01', periods=24, freq='MS')
y = np.random.rand(24).cumsum()
ax.plot(x, y, marker='o', color='steelblue')

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.get_xticklabels(), rotation=0, ha='center', fontsize=9)
ax.tick_params(axis='y', labelsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.set_title('Quarterly tick density')
plt.tight_layout()
plt.show()

## 4. Grids — supporting lines, not the main event

Grids help compare values. Make them quiet: low alpha, light gray, axis-specific
(`axis='y'` is usually enough).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for i, s in enumerate([None, 'y', 'both']):
    axes[i].plot([1, 2, 3, 4], [10, 14, 12, 18], marker='o', color='steelblue')
    if s:
        axes[i].grid(axis=s, color='gray', alpha=0.3)
    axes[i].spines[['top', 'right']].set_visible(False)
    axes[i].set_title(f'grid axis={s or "none"}')
plt.tight_layout()
plt.show()

## 5. `rcParams` — the global styling layer

Every matplotlib default lives in `plt.rcParams`. Setting them once at the top
of a notebook changes every subsequent chart.

In [ ]:
import matplotlib as mpl

print('Some rcParams you will touch a lot:')
for k in ['figure.figsize', 'figure.dpi', 'savefig.dpi', 'font.size',
          'axes.titlesize', 'axes.labelsize', 'axes.spines.top',
          'axes.spines.right', 'axes.grid', 'xtick.labelsize',
          'ytick.labelsize', 'lines.linewidth', 'figure.facecolor']:
    print(f'  {k:25s} = {mpl.rcParams[k]!r}')

In [ ]:
plt.rcParams.update({
    'figure.figsize'   : (8, 4),
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'lines.linewidth'  : 2.0,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
x = np.linspace(0, 4*np.pi, 100)
axes[0].plot(x, np.sin(x), color='steelblue'); axes[0].set_title('sin')
axes[1].plot(x, np.cos(x), color='crimson');  axes[1].set_title('cos')
plt.tight_layout()
plt.show()

## 6. Style sheets — one-line restyling

Style sheets bundle many `rcParams` together. Useful built-ins:

- `plt.style.use('default')`
- `plt.style.use('seaborn-v0_8-whitegrid')`
- `plt.style.use('fivethirtyeight')`
- `plt.style.use('ggplot')`
- `plt.style.use('tableau-colorblind10')`

Always reset to `default` after experimenting so you don't leak styles into
later cells.

In [ ]:
styles = ['default', 'seaborn-v0_8-whitegrid', 'fivethirtyeight', 'ggplot']
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
x = np.linspace(0, 6, 100)
for ax, style in zip(axes.ravel(), styles):
    with plt.style.context(style):
        ax.plot(x, np.sin(x), label='sin')
        ax.plot(x, np.cos(x), label='cos')
        ax.set_title(style)
        ax.legend()
plt.tight_layout()
plt.show()

## 7. Legends — placement and styling

Best practice for legends:

1. Drop the frame: `frameon=False`
2. Place them where they do not block data
3. If only one series, sometimes a label in the title is better than a legend

In [ ]:
np.random.seed(0)
x = np.linspace(0, 6, 100)
y1 = np.sin(x) + np.random.normal(0, 0.15, 100)
y2 = np.cos(x) + np.random.normal(0, 0.15, 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, y1, label='series A')
axes[0].plot(x, y2, label='series B')
axes[0].legend(loc='upper right')
axes[0].set_title('Default — boxed legend')

axes[1].plot(x, y1, label='series A')
axes[1].plot(x, y2, label='series B')
axes[1].legend(loc='lower left', frameon=False)
axes[1].set_title('Frame off')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Titles that earn their space

A title should answer the *headline*, not describe the data. Compare:

- ❌ "MRR over time (2024)"
- ✅ "MRR grew 42% in 2024, driven by EMEA expansion"

Pair titles with a *subtitle* using `fig.suptitle` + `ax.set_title`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = pd.date_range('2024-01-01', periods=12, freq='MS')
y = 100 * (1.04 ** np.arange(12)) + np.random.normal(0, 2, 12)
ax.plot(x, y, marker='o', color='steelblue')

fig.suptitle('MRR grew 47% in 2024', fontsize=14, weight='bold', x=0.05, ha='left')
ax.set_title('Monthly recurring revenue, Jan–Dec 2024',
             fontsize=10, color='gray', loc='left', pad=10)
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylabel('USD millions')
plt.tight_layout()
plt.show()

## 9. Building a reusable style context

Once you have a style you like, wrap it in a function so the whole notebook
(or a whole report) inherits it.

In [ ]:
def set_style():
    plt.rcParams.update({
        'figure.figsize'   : (8, 4),
        'figure.dpi'       : 110,
        'savefig.dpi'      : 150,
        'axes.spines.top'  : False,
        'axes.spines.right': False,
        'axes.grid'        : True,
        'grid.color'       : 'lightgray',
        'grid.alpha'       : 0.5,
        'axes.titlesize'   : 13,
        'axes.titleweight' : 'bold',
        'axes.titlepad'    : 12,
        'axes.labelsize'   : 10,
        'xtick.labelsize'  : 9,
        'ytick.labelsize'  : 9,
        'lines.linewidth'  : 2.0,
        'font.family'      : 'DejaVu Sans',
    })

set_style()
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [1, 4, 9], marker='o')
ax.set_title('Style applied')
plt.show()

## Summary

| Concept | Key point |
|---|---|
| Spines | Hide top & right; lighten the rest |
| Ticks | Use `Locators` + `Formatters` for dates/percents |
| Grid | Quiet: low alpha, axis='y' usually enough |
| rcParams | Global; set once at the top of the notebook |
| Style sheets | One-line restyling; use `with plt.style.context()` to scope |
| Legends | `frameon=False`, place where data permits |
| Titles | Headline (finding), then subtitle (what) |

**Next:** `04_univariate_distributions.ipynb` — the first chart family.